<a href="https://colab.research.google.com/github/Hemanth4249/irs-exp5/blob/main/Exp_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
rom sklearn.datasets import fetch_20newsgroups
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import confusion_matrix
import numpy as np

# Load a subset of the 20 newsgroups dataset (for speed)
categories = ['alt.atheism', 'comp.graphics', 'sci.space']
newsgroups = fetch_20newsgroups(subset='all', categories=categories, remove=('headers', 'footers', 'quotes'))
# Vectorize documents using TF-IDF
vectorizer = TfidfVectorizer(stop_words='english', max_df=0.5)
X = vectorizer.fit_transform(newsgroups.data)
# Number of clusters = number of categories
k = len(categories)
# K-means clustering
kmeans = KMeans(n_clusters=k, random_state=42)
kmeans.fit(X)
labels_pred = kmeans.labels_


# Ground truth labels
labels_true = newsgroups.target

# Function to compute Purity
def purity_score(y_true, y_pred):
    # Compute contingency matrix (also called confusion matrix)
    contingency_matrix = confusion_matrix(y_true, y_pred)
    return np.sum(np.amax(contingency_matrix, axis=0)) / np.sum(contingency_matrix)

# Compute Precision, Recall, F-measure for each cluster
def precision_recall_fmeasure(y_true, y_pred):
    unique_clusters = np.unique(y_pred)
    precision_list = []
    recall_list = []
    fmeasure_list = []

    for cluster in unique_clusters:
        # Find documents assigned to this cluster
        cluster_indices = np.where(y_pred == cluster)[0]
        # Most common true class in this cluster
        true_labels_in_cluster = y_true[cluster_indices]
        if len(true_labels_in_cluster) == 0:
            continue
        dominant_class = np.bincount(true_labels_in_cluster).argmax()

        # True Positives (TP): docs in cluster and true class = dominant_class
        TP = np.sum(true_labels_in_cluster == dominant_class)
        # False Positives (FP): docs in cluster but not dominant_class
        FP = len(true_labels_in_cluster) - TP
        # False Negatives (FN): docs in dominant_class but not in this cluster
        FN = np.sum((y_true == dominant_class)) - TP

        precision = TP / (TP + FP) if (TP + FP) > 0 else 0
        recall = TP / (TP + FN) if (TP + FN) > 0 else 0
        fmeasure = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

        precision_list.append(precision)
        recall_list.append(recall)
        fmeasure_list.append(fmeasure)

    # Macro-average over clusters
    return np.mean(precision_list), np.mean(recall_list), np.mean(fmeasure_list)

# Calculate metrics
purity = purity_score(labels_true, labels_pred)
precision, recall, fmeasure = precision_recall_fmeasure(labels_true, labels_pred)

print(f"Purity: {purity:.4f}")
print(f"Precision (macro-average): {precision:.4f}")
print(f"Recall (macro-average): {recall:.4f}")
print(f"F-measure (macro-average): {fmeasure:.4f}")

Purity: 0.6897
Precision (macro-average): 0.8185
Recall (macro-average): 0.6731
F-measure (macro-average): 0.6808
